In [5]:
import pandas as pd

df = pd.read_csv('SampleSuperstore.csv', encoding='windows-1252')
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [6]:
df.shape

(9994, 21)

In [8]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

In [10]:
df.dtypes

Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

In [11]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

In [12]:
df.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df['Delivery Days'] = (df['Ship Date'] - df['Order Date']).dt.days

In [15]:
df[['Sales','Profit','Discount','Quantity']].describe()

,Sales,Profit,Discount,Quantity
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,28.656896,0.156203,3.789574
std,623.245101,234.260108,0.206452,2.225110
min,0.444000,-6599.978000,0.000000,1.000000
25%,17.280000,1.728750,0.000000,2.000000
50%,54.490000,8.666500,0.200000,3.000000
75%,209.940000,29.364000,0.200000,5.000000
max,22638.480000,8399.976000,0.800000,14.000000


In [16]:
df[df['Profit'] < 0].shape

(1871, 22)

In [17]:
df['High Discount'] = df['Discount'] > 0.2

In [18]:
df['Is Loss'] = df['Profit'] < 0

In [19]:
#high sales threshold
threshold = df['Sales'].quantile(0.75)
threshold

np.float64(209.94)

In [43]:
df['High Sales'] = df['Sales'] > threshold


In [21]:
df['High Sales Loss'] = (df['High Sales']) & (df['Is Loss'])
df[df['High Sales Loss']].shape

(584, 26)

In [22]:
df.groupby('High Discount')['Profit'].mean()

High Discount
False    49.037679
True    -97.183098
Name: Profit, dtype: float64

In [23]:
df.groupby('High Discount')['Is Loss'].mean()

High Discount
False    0.060807
True     0.967696
Name: Is Loss, dtype: float64

In [24]:
df.groupby('Category')[['Sales','Profit','Discount']].mean()

,Sales,Profit,Discount
Category,,,
Furniture,349.834887,8.699327,0.173923
Office Supplies,119.324101,20.327050,0.157285
Technology,452.709276,78.752002,0.132323


In [25]:
df.groupby('Sub-Category')[['Sales','Profit','Discount']].mean().sort_values('Profit')

,Sales,Profit,Discount
Sub-Category,,,
Tables,648.794771,-55.565771,0.261285
Bookcases,503.859633,-15.230509,0.211140
Supplies,245.650200,-6.258418,0.076842
Fasteners,13.936774,4.375660,0.082028
Art,34.068834,8.200737,0.074874
Furnishings,95.825668,13.645918,0.138349
Labels,34.303055,15.236962,0.068681
Binders,133.560560,19.843574,0.372292
Paper,57.284092,24.856620,0.074891


In [26]:
df.groupby('Sub-Category')['Is Loss'].sum().sort_values(ascending=False)

Sub-Category
Binders        613
Chairs         235
Tables         203
Furnishings    167
Storage        161
Phones         136
Bookcases      109
Accessories     91
Appliances      67
Machines        44
Supplies        33
Fasteners       12
Art              0
Envelopes        0
Copiers          0
Paper            0
Labels           0
Name: Is Loss, dtype: int64

In [27]:
df.groupby('Sub-Category')['High Discount'].mean().sort_values(ascending=False)

Sub-Category
Tables         0.551724
Machines       0.460870
Binders        0.402495
Bookcases      0.307018
Chairs         0.256078
Furnishings    0.144201
Appliances     0.143777
Copiers        0.132353
Phones         0.122610
Accessories    0.000000
Fasteners      0.000000
Art            0.000000
Envelopes      0.000000
Paper          0.000000
Labels         0.000000
Storage        0.000000
Supplies       0.000000
Name: High Discount, dtype: float64

In [28]:
df.groupby('Region')[['Sales','Profit','Discount']].mean()

,Sales,Profit,Discount
Region,,,
Central,215.772661,17.092709,0.240353
East,238.336110,32.135808,0.145365
South,241.803645,28.857673,0.147253
West,226.493233,33.849032,0.109335


In [29]:
df.groupby('Region')['Is Loss'].mean()

Region
Central    0.318984
East       0.194171
South      0.159877
West       0.099282
Name: Is Loss, dtype: float64

In [30]:
df.groupby('Segment')[['Sales','Profit','Discount']].mean()

,Sales,Profit,Discount
Segment,,,
Consumer,223.733644,25.836873,0.158141
Corporate,233.823300,30.456667,0.158228
Home Office,240.972041,33.818664,0.147128


In [34]:
df['Discount Bucket'] = pd.cut(df['Discount'], bins=[0,0.1,0.2,0.3,0.5,1],
                              labels=['0-10','10-20','20-30','30-50','50+'])

In [35]:
df.groupby('Discount Bucket')['Profit'].mean()

C:\Users\lenovo\AppData\Local\Temp\ipykernel_12160\1257226420.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('Discount Bucket')['Profit'].mean()


Discount Bucket
0-10      96.055074
10-20     24.738824
20-30    -45.679636
30-50   -156.282991
50+      -89.438144
Name: Profit, dtype: float64

In [ ]:
#High Quantity Low Profit
df[(df['Quantity'] > 5) & (df['Profit'] < 10)].shape

(640, 27)

In [41]:
problem_orders = df[(df['High Discount']) & (df['Is Loss'])]
problem_orders.shape

(1348, 27)

1. **Introduce Discount Caps**
   - Limit discounts till 20%

2. **Reprice Loss-making Products**
   - Especially Furniture sub-categories

3. **Improve Logistics**
   - Reduce delivery delays to avoid compensation discounts
